In [49]:
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras import Model

In [50]:
import numpy as np

X = np.load("../data/X.npy")

print(X.shape)
print(X.dtype)

(3521, 300, 90)
float64


In [51]:
y_act = np.load("../data/y_act.npy")

print(y_act.shape)

print(np.unique(y_act))

(3521,)
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]


In [52]:
X_train_clean = np.load("X_train_clean.npy")
y_train_clean = np.load("y_train_clean.npy")

X_test_clean = np.load("X_test_clean.npy")
y_test_clean = np.load("y_test_clean.npy")

print(X_train_clean.shape)
print(X_test_clean.shape)

(1580, 128, 90)
(395, 128, 90)


In [53]:
WIAR_MAP = {
    11:1,   # walk
    10:2,   # hand_clap
    0:3,    # horizontal_arm_wave
    2:4     # two_hands_wave
}

In [54]:
selected_idx = np.isin(
    y_act,
    list(WIAR_MAP.keys())
)

X_wiar = X[selected_idx]

y_wiar_raw = y_act[selected_idx]

print(X_wiar.shape)
print(y_wiar_raw.shape)

(880, 300, 90)
(880,)


In [55]:
y_wiar = np.array([
    WIAR_MAP[y]
    for y in y_wiar_raw
])

print(np.unique(y_wiar))

[1 2 3 4]


In [56]:
X_wiar_norm = (
    X_wiar -
    X_wiar.mean(
        axis=(1,2),
        keepdims=True
    )
) / (
    X_wiar.std(
        axis=(1,2),
        keepdims=True
    ) + 1e-8
)

print("WIAR normalize tamam ")

WIAR normalize tamam 


In [57]:
from sklearn.model_selection import train_test_split

X_wiar_train, X_wiar_test, y_wiar_train, y_wiar_test = train_test_split(
    X_wiar_norm,
    y_wiar,
    test_size=0.2,
    random_state=42,
    stratify=y_wiar
)

print(X_wiar_train.shape)
print(X_wiar_test.shape)

(704, 300, 90)
(176, 300, 90)


In [58]:
inputs = layers.Input(shape=(300,90))

x = layers.Conv1D(
    64,
    kernel_size=5,
    activation='relu',
    padding='same'
)(inputs)

x = layers.BatchNormalization()(x)

x = layers.MaxPooling1D(2)(x)

x = layers.Dropout(0.3)(x)

x = layers.Conv1D(
    128,
    kernel_size=3,
    activation='relu',
    padding='same'
)(x)

x = layers.BatchNormalization()(x)

x = layers.MaxPooling1D(2)(x)

x = layers.Dropout(0.3)(x)

x = layers.GlobalAveragePooling1D()(x)

feature_layer = layers.Dense(
    128,
    activation='relu',
    name="feature_layer"
)(x)

x = layers.Dropout(0.3)(feature_layer)

outputs = layers.Dense(
    5,
    activation='softmax'
)(x)

wiar_model = Model(inputs, outputs)

wiar_model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 300, 90)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_8 (Conv1D)               │ (None, 300, 64)        │        28,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 300, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_8 (MaxPooling1D)  │ (None, 150, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 150, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_9 (Conv1D)               │ (None, 150, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 150, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_9 (MaxPooling1D)  │ (None, 75, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 75, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_4      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_layer (Dense)           │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71,493 (279.27 KB)

 Trainable params: 71,109 (277.77 KB)

 Non-trainable params: 384 (1.50 KB)

In [59]:
wiar_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [60]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

In [61]:
history = wiar_model.fit(
    X_wiar_train,
    y_wiar_train,
    validation_data=(
        X_wiar_test,
        y_wiar_test
    ),
    epochs=30,
    batch_size=64,
    callbacks=[early_stop]
)

Epoch 1/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 59ms/step - accuracy: 0.4347 - loss: 1.3803 - val_accuracy: 0.4432 - val_loss: 1.3287
Epoch 2/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - accuracy: 0.6676 - loss: 0.8596 - val_accuracy: 0.5000 - val_loss: 1.1245
Epoch 3/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.7699 - loss: 0.6474 - val_accuracy: 0.6761 - val_loss: 0.9620
Epoch 4/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.8537 - loss: 0.4861 - val_accuracy: 0.8182 - val_loss: 0.8275
Epoch 5/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.8878 - loss: 0.3888 - val_accuracy: 0.8580 - val_loss: 0.7248
Epoch 6/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.9006 - loss: 0.3226 - val_accuracy: 0.8693 - val_loss: 0.6277
Epoch 7/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.9077 - loss: 0.2938 - val_accuracy: 0.9261 - val_loss: 0.5485
Epoch 8/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 48ms/step - accuracy: 0.9006 - loss: 0.3009 - val_accuracy: 0.9148 - v

In [62]:
feature_extractor = Model(
    inputs=wiar_model.input,
    outputs=wiar_model.get_layer(
        "feature_layer"
    ).output
)

In [63]:
inputs = layers.Input(shape=(128,90))

x = layers.Conv1D(
    64,
    kernel_size=5,
    activation='relu',
    padding='same'
)(inputs)

x = layers.BatchNormalization()(x)

x = layers.MaxPooling1D(2)(x)

x = layers.Dropout(0.3)(x)

x = layers.Conv1D(
    128,
    kernel_size=3,
    activation='relu',
    padding='same'
)(x)

x = layers.BatchNormalization()(x)

x = layers.MaxPooling1D(2)(x)

x = layers.Dropout(0.3)(x)

x = layers.GlobalAveragePooling1D()(x)

feature_layer = layers.Dense(
    128,
    activation='relu',
    name="feature_layer"
)(x)

x = layers.Dropout(0.3)(feature_layer)

outputs = layers.Dense(
    5,
    activation='softmax'
)(x)

transfer_model = Model(inputs, outputs)

In [64]:
for i in range(len(transfer_model.layers)):

    try:

        transfer_model.layers[i].set_weights(
            wiar_model.layers[i].get_weights()
        )

    except:

        pass


In [65]:
transfer_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [66]:
history = transfer_model.fit(
    X_train_clean,
    y_train_clean,
    validation_data=(
        X_test_clean,
        y_test_clean
    ),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.4709 - loss: 1.4747 - val_accuracy: 0.3165 - val_loss: 2.0731
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6778 - loss: 0.8272 - val_accuracy: 0.4481 - val_loss: 1.8420
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7627 - loss: 0.6663 - val_accuracy: 0.6658 - val_loss: 1.3066
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8025 - loss: 0.5342 - val_accuracy: 0.5797 - val_loss: 1.4372
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8209 - loss: 0.4659 - val_accuracy: 0.6127 - val_loss: 1.2886


In [67]:
y_pred_probs = transfer_model.predict(
    X_test_clean
)

y_pred = np.argmax(
    y_pred_probs,
    axis=1
)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


In [68]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test_clean,
    y_pred
)

print(cm)

[[16 22  2  2 38]
 [ 0 15  0  8 57]
 [ 0 22  5  4 49]
 [ 0 24  0 24 27]
 [ 0 15  0  0 65]]


In [70]:
ACTIVITIES = [
    "still",
    "walk",
    "hand_clap",
    "horizontal_arm_wave",
    "two_hands_wave"
]

In [71]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_clean,
        y_pred,
        target_names=ACTIVITIES
    )
)

                     precision    recall  f1-score   support

              still       1.00      0.20      0.33        80
               walk       0.15      0.19      0.17        80
          hand_clap       0.71      0.06      0.11        80
horizontal_arm_wave       0.63      0.32      0.42        75
     two_hands_wave       0.28      0.81      0.41        80

           accuracy                           0.32       395
          macro avg       0.55      0.32      0.29       395
       weighted avg       0.55      0.32      0.29       395

